In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [3]:
import numpy as np

from datasets import load_from_disk

from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [4]:
bias_dataset = load_from_disk(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/tokenized_bias"
)

bias_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 29258
})

In [5]:
dataset = bias_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]

eval_dataset = dataset["test"]

In [6]:
MODEL_NAME = "roberta-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [8]:
training_args = TrainingArguments(

    output_dir="./bias_model",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    fp16=True,

    logging_steps=100,

    report_to="none"
)

In [9]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    compute_metrics=compute_metrics
)

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.445839,0.471147,0.821599,0.825348,0.821599,0.822171
2,0.408068,0.449735,0.855434,0.860111,0.855434,0.855063
3,0.291465,0.574617,0.866029,0.866816,0.866029,0.865962


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8778, training_loss=0.4408170157289364, metrics={'train_runtime': 2396.8613, 'train_samples_per_second': 29.296, 'train_steps_per_second': 3.662, 'total_flos': 1.8475297966024704e+16, 'train_loss': 0.4408170157289364, 'epoch': 3.0})

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer.save_pretrained(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/bias_model"
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

('/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/bias_model/tokenizer_config.json',
 '/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/bias_model/tokenizer.json')

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/bias_model"
)

print("Tokenizer loaded successfully!")
print(type(tokenizer))

Tokenizer loaded successfully!
<class 'transformers.models.roberta.tokenization_roberta.RobertaTokenizer'>


In [11]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.291465,0.574617,3,0.866029,0.866816,0.866029,0.865962


{'eval_loss': 0.5746170878410339,
 'eval_accuracy': 0.8660287081339713,
 'eval_precision': 0.866815575626868,
 'eval_recall': 0.8660287081339713,
 'eval_f1': 0.8659621726521464}

In [12]:
trainer.save_model(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/bias_model"
)

print("Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!
